# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print dataset name and description
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Here, we print out all record set `@id`s, their field names, and associated field `@id`s in the dataset.

In [ ]:
# Print available record sets, their field names and field @ids
if hasattr(metadata, 'record_sets'):
    print("Available record sets:")
    for rs in metadata.record_sets:
        print(f"  Record Set '@id': {rs.id}, name: {getattr(rs, 'name', None)}")
        if hasattr(rs, 'fields') and rs.fields:
            print("    Fields:")
            for field in rs.fields:
                print(f"      Field '@id': {field.id}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All entities are referenced by their `@id`.

Here, we extract data from each discovered record set. Adjust the record set and field `@id` below as needed.

In [ ]:
# Prepare record set @ids
if hasattr(metadata, 'record_sets'):
    record_set_ids = [rs.id for rs in metadata.record_sets]
else:
    record_set_ids = []

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for Record Set '@id': {record_set_id}")
        print(f"Columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"Could not load records for Record Set '@id': {record_set_id}. Error: {e}")

# Example: print top rows for the first available record set
if dataframes:
    first_rs = record_set_ids[0]
    print(f"\nFirst rows of Record Set '@id': {first_rs}")
    display(dataframes[first_rs].head())
else:
    print('No dataframes loaded. Please check the record set availability.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes sample operations on a numeric field and grouping by a key attribute for analysis.

_Make sure to select the field `@id` from the previous overview. Adjust the numeric field, threshold, and group field for your analysis._

In [ ]:
# If there is at least one record set loaded:
if dataframes:
    # Choose the first record set for demonstration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    
    # Attempt to find a numeric field
    numeric_field = None
    group_field = None
    
    # Infer types by pandas dtype
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    
    # Also select a possible group field (categorical)
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break
    
    if numeric_field:
        print(f"Using numeric field '@id': {numeric_field}")
        threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with '{numeric_field}' > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        if group_field and group_field in filtered_df.columns:
            print(f"\nGrouping filtered data by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df)
    else:
        print("No numeric field found for analysis.")
else:
    print("No dataframes to analyze. Please ensure successful data extraction.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Visualization for the numeric field in the first dataframe
if dataframes and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20, color="teal")
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field or data available for visualization.')

## 6. Conclusion
In this notebook, we loaded and examined the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya with `mlcroissant`:

- Inspected dataset record sets, discovered fields and their `@id`s.
- Extracted records into pandas DataFrames for further exploration.
- Performed simple filtering, normalization, and grouping on a numeric field.
- Visualized distributions and group averages for key fields.

Further steps may include hypothesis testing, building predictive models, handling missing values more robustly, and exploring relationships between mental health indicators and demographic variables.